# Wikidata Ogham Sites — Overview

## About this notebook

This notebook is a hands-on introduction to working with **knowledge
graph data** in Python, using [Wikidata](https://www.wikidata.org) —
the largest openly-licensed general-purpose knowledge graph — as a
live data source. It retrieves records about Ogham stones (early
medieval inscribed monuments found mainly in Ireland and western
Britain) via a [SPARQL](https://www.w3.org/TR/sparql11-query/) query
and visualises their distribution as bar charts.

It is part of an Open Educational Resource (OER) series on knowledge
graphs and linked open data, and is designed to stand on its own:
you do not need to have read any other notebook in the series to
follow along. A browser-executable variant of this notebook is
available as `wikidata-ogham-sites-live.qmd` (same content, no
installation required).

### Why this dataset?

Ogham stones are a useful teaching dataset because they are:

- **Well-curated** on Wikidata, with typed instances, find-spots, and
  administrative districts linked by dedicated properties.
- **Small enough** to fit comfortably in memory but large enough to
  yield meaningful aggregations.
- **Rich in structure**: each record participates in several
  relationships (instance-of, find-spot, county), which makes them a
  good example for both entity-centric and aggregation queries.

The same pipeline — SPARQL query → DataFrame → visualisation — can
be reused for any other knowledge-graph dataset, regardless of
domain.

### Tooling notes

- **`SPARQLWrapper`** is used here to query Wikidata. It is the
  most common Python client for SPARQL endpoints and handles the
  `Accept`-header negotiation and JSON parsing for you. The
  browser-executable variant (`-live.qmd`) uses `pyodide.http.pyfetch`
  instead, because `SPARQLWrapper` depends on blocking HTTP which
  does not work in Pyodide.
- **`pandas`** to hold results in a tabular form. Once data is in a
  DataFrame, standard data-science tooling applies — regardless of
  the original source being a graph.
- **`matplotlib`** for static bar charts.

### Requirements

```bash
pip install SPARQLWrapper pandas matplotlib
```

## Step 1 — Defining the SPARQL query

The query below asks Wikidata for every item that is an instance of
[Ogham stone](https://www.wikidata.org/wiki/Q2016147) (`wd:Q2016147`),
together with its find-spot (linked via `wdt:P189`), the county in
which the find-spot lies, and — optionally — its coordinate location
(`wdt:P625`).

Two notes on query design that generalise to other knowledge-graph
queries:

- **`SERVICE wikibase:label`** is a Wikidata-specific service that
  returns human-readable labels for every item variable that also has
  a `?…Label` companion in the `SELECT` clause. It is significantly
  cheaper than joining `rdfs:label` manually.
- **`OPTIONAL`** is used for coordinates here because not every stone
  in Wikidata has them; making them mandatory would drop perfectly
  valid records. The map variant of this notebook makes the opposite
  choice, because records without coordinates cannot be plotted.

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams["figure.dpi"] = 120

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "OER-Notebook/1.0 (NFDI4Objects)"

SPARQL_QUERY = """
SELECT ?item ?itemLabel ?geo ?site ?siteLabel ?county ?countyLabel WHERE {
  ?item wdt:P31 wd:Q2016147.
  ?item wdt:P189 ?site.
  ?site wdt:P31 wd:Q72617071.
  ?item wdt:P189 ?county.
  ?county wdt:P31 wd:Q179872.
  OPTIONAL { ?item wdt:P625 ?geo. }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

def fetch_wikidata(query, endpoint=SPARQL_ENDPOINT):
    """Send a SPARQL query to the given endpoint and return the raw bindings."""
    sparql = SPARQLWrapper(endpoint, agent=USER_AGENT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    return sparql.queryAndConvert()["results"]["bindings"]

print("✓ Helpers defined.")

## Step 2 — Loading the data

SPARQL results come back as JSON in a format called *bindings*: a
list of dictionaries, one per solution, where each key maps to a
`{"type": ..., "value": ...}` object. We flatten these into plain
records and build a DataFrame. This shape — flat records with
consistent keys — is almost always what you want when you plan to
plot or aggregate.

> **Common pitfall: duplicates.** A single Ogham stone can appear in
> multiple rows because a stone may be linked to more than one
> "find-spot" in Wikidata (e.g. both the original location and a
> current museum). When computing aggregates, remember to use
> `nunique()` or `drop_duplicates()` where appropriate, rather than
> `len(df)`.

In [ ]:
results = fetch_wikidata(SPARQL_QUERY)

rows = []
for r in results:
    rows.append({
        "item":        r["item"]["value"],
        "itemLabel":   r["itemLabel"]["value"],
        "geo":         r.get("geo", {}).get("value"),
        "site":        r["site"]["value"],
        "siteLabel":   r["siteLabel"]["value"],
        "county":      r["county"]["value"],
        "countyLabel": r["countyLabel"]["value"],
    })

df = pd.DataFrame(rows)
print(f"✓ {len(df)} records loaded — {df['itemLabel'].nunique()} unique Ogham stones")
df.head()

## Step 3a — Visualisation: top two find-spots per county

For each Irish county, we identify the two find-spots with the
highest number of associated Ogham stones. This highlights
concentration patterns — which is often the first thing a domain
expert wants to see when exploring a new dataset.

In [ ]:
# One colour per county, used consistently across plots in this notebook.
unique_counties = df["countyLabel"].unique()
county_colors = {
    county: color
    for county, color in zip(unique_counties, plt.cm.tab20.colors)
}

# For each county, keep the two find-spots with the highest counts.
# Using sort_values + head inside groupby avoids pandas' apply+nlargest
# quirk around grouping-column inclusion — and is arguably clearer too.
site_counts = (
    df.groupby(["countyLabel", "siteLabel"])
      .size()
      .reset_index(name="count")
)
top_sites = (
    site_counts
      .sort_values(["countyLabel", "count"], ascending=[True, False])
      .groupby("countyLabel", as_index=False)
      .head(2)
      .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 7))
for county, group in top_sites.groupby("countyLabel"):
    ax.bar(group["siteLabel"], group["count"],
           color=county_colors[county], label=county)

ax.set_title("Top 2 find-spots per county with the most Ogham stones")
ax.set_xlabel("Find-spot")
ax.set_ylabel("Number of Ogham stones")
plt.xticks(rotation=45, ha="right")
ax.legend(title="County", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Step 3b — Visualisation: distribution by county

A simpler aggregation: how many Ogham-stone records does each county
have? This kind of plot is the sanity-check step of almost any
knowledge-graph query — if one county dominates the counts
implausibly, that often signals a data-modelling quirk rather than a
real-world pattern.

In [ ]:
county_counts = df["countyLabel"].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
county_counts.plot(
    kind="bar",
    color=[county_colors[c] for c in county_counts.index],
    ax=ax,
)
ax.set_title("Distribution of Ogham stones by county")
ax.set_xlabel("County")
ax.set_ylabel("Number of Ogham stones")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 4 — Exploring the data

The cell below is a free playground. Edit the `county_filter` value,
or write your own aggregations — the DataFrame `df` is available for
the rest of the session.

In [ ]:
# Example: list all stones from a specific county — change the filter here.
county_filter = "County Kerry"
subset = (
    df[df["countyLabel"] == county_filter][["itemLabel", "siteLabel"]]
      .drop_duplicates()
)
print(f"Ogham stones in {county_filter}: {len(subset)}")
subset

---

*Part of an Open Educational Resource series on knowledge graphs and
linked open data, produced in the context of
[NFDI4Objects](https://www.nfdi4objects.net/).*